# TrueBeat: train the face-swap detector (docs/12)

**What this does.** Downloads FaceForensics++ c23 (original + Deepfakes, Face2Face, FaceSwap,
NeuralTextures, FaceShifter) and a DeepFakeDetection subset **into this Colab runtime** (never your
laptop), cuts the face out of 16 frames per video, embeds the crops with three candidate vision
models on the GPU, picks one by the rule fixed in `docs/12` §6 (on train + val only), and
evaluates it once on the held-out test set and on DeepFakeDetection. The result is one small file,
`faceswap_result.zip`, on your Google Drive.

**How to run.** Runtime → Change runtime type → **T4 GPU** (or better) → **Run all**.
About 1–2 hours. If Colab disconnects, run all again: downloads and features resume from Drive.

**Terms.** FaceForensics++ data is for non-commercial research only
([terms](http://kaldir.vc.in.tum.de/faceforensics/webpage/FaceForensics_TOS.pdf)); this project
already accepted them. Only these public research videos are processed here: nothing is uploaded
from your laptop.

**After it finishes.** Download `MyDrive/trubeat_faceswap/faceswap_result.zip` into
`~/Projects/trubeat/runs/faceswap/` and tell the agent.

In [ ]:
# Settings
REPO_URL = "https://github.com/luckyDuckBuck/trubeat.git"
BRANCH = "round3-faceswap"          # the branch with the face-swap code
DRIVE_DIR = "/content/drive/MyDrive/trubeat_faceswap"
DATA = "/content/ffpp"               # runtime disk (fast; wiped when the runtime ends)
FEAT = "/content/features"
CANDIDATES = ["vit_small_patch14_dinov2.lvd142m", "vit_base_patch16_clip_224.openai",
              "vit_base_patch14_dinov2.lvd142m"]
CAND = " ".join(CANDIDATES)
DFD_REAL, DFD_FAKE = 150, 300
import os, subprocess
WORKERS = max(1, (os.cpu_count() or 2) - 1)
print("CPU workers:", WORKERS)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# Code + an isolated environment with the project's pinned versions (numpy<2, mediapipe 0.10.21),
# so Colab's own packages are untouched and no runtime restart is needed.
if not os.path.isdir("/content/trubeat"):
    !git clone -q --branch {BRANCH} {REPO_URL} /content/trubeat
else:
    !git -C /content/trubeat pull -q
!pip -q install uv
if not os.path.exists("/content/venv/bin/python"):
    !uv venv -q /content/venv --python 3.12
!uv pip install -q -p /content/venv/bin/python torch torchvision
!uv pip install -q -p /content/venv/bin/python -e "/content/trubeat[aigen]"
PY = "/content/venv/bin/python"
TB = "/content/venv/bin/trubeat"
!{PY} -c "import torch, timm, mediapipe, numpy; print('cuda', torch.cuda.is_available(), 'numpy', numpy.__version__)"
for b in CANDIDATES:
    !{TB} fetch-models --backbone {b}
%cd /content/trubeat

In [ ]:
# 1. Download (resumable; ~16 GB to the runtime disk, typically 10-30 min)
!{PY} scripts/train_faceswap.py download --root {DATA} --dfd-real {DFD_REAL} --dfd-fake {DFD_FAKE} --threads 8

In [ ]:
# 2. Manifest (official identity-disjoint splits; DeepFakeDetection is external only)
!{PY} scripts/train_faceswap.py manifest --root {DATA} --out {DRIVE_DIR}/manifest.csv

In [ ]:
# 3. Face crops + embeddings for every candidate (GPU), plus 380 px crops for stage 2.
#    Checkpointed to Drive every 1500 videos; a rerun after a disconnect resumes.
!{PY} scripts/train_faceswap.py extract --manifest {DRIVE_DIR}/manifest.csv --features-dir {FEAT} \
    --backbones {CAND} --workers {WORKERS} --device cuda --save-crops {FEAT}/crops \
    --checkpoint-tar {DRIVE_DIR}/features.tar --checkpoint-every 1500

In [ ]:
# 4. Selection on train+val, threshold, held-out test + DeepFakeDetection, the docs/12 gates
!{PY} scripts/train_faceswap.py select --manifest {DRIVE_DIR}/manifest.csv --features-dir {FEAT} \
    --out-dir {DRIVE_DIR}/result --backbones {CAND}

In [ ]:
# 4b. Stage 2 (docs/12 §3), only if stage 1 failed gate 1: fine-tune EfficientNet-B4 on the crops
gate1 = subprocess.run([PY, "-c", (f"import joblib; g = joblib.load('{DRIVE_DIR}/result/faceswap.joblib')['gates']; "
                                   "print(all(ok is not False for n, ok, v in g if n.startswith('1.')))")],
                       capture_output=True, text=True).stdout.strip()
print("stage 1 passes gate 1:", gate1)
STAGE2 = gate1 != "True"
if STAGE2:
    !{PY} scripts/train_faceswap.py finetune --manifest {DRIVE_DIR}/manifest.csv --crops-dir {FEAT}/crops \
        --out-dir {DRIVE_DIR}/result --workers {WORKERS} --epochs 6 --batch 24

In [ ]:
# 5. Robustness: the held-out videos after a WhatsApp-like re-encode (640 px wide, CRF 30)
best = subprocess.run([PY, "-c", f"import joblib; print(joblib.load('{DRIVE_DIR}/result/faceswap.joblib')['config']['backbone'])"],
                      capture_output=True, text=True).stdout.strip()
print("stage-1 backbone:", best)
!{PY} scripts/train_faceswap.py extract --manifest {DRIVE_DIR}/manifest.csv --features-dir /content/features_whatsapp \
    --backbones {best} --workers {WORKERS} --device cuda --transform whatsapp --splits test external \
    --save-crops /content/features_whatsapp/crops
!{PY} scripts/train_faceswap.py evaluate --manifest {DRIVE_DIR}/manifest.csv --features-dir /content/features_whatsapp \
    --bundle {DRIVE_DIR}/result/faceswap.joblib --report {DRIVE_DIR}/result/faceswap_report.md \
    --title "Stage 1 robustness: WhatsApp-like re-encode (640 px wide, CRF 30)"
if STAGE2:
    !{PY} scripts/train_faceswap.py evaluate --manifest {DRIVE_DIR}/manifest.csv --crops-dir /content/features_whatsapp/crops \
        --bundle {DRIVE_DIR}/result/faceswap_cnn.joblib --report {DRIVE_DIR}/result/faceswap_report.md --device cuda \
        --title "Stage 2 robustness: WhatsApp-like re-encode (640 px wide, CRF 30)"

In [ ]:
# 6. One file to bring back to the laptop
!cd {DRIVE_DIR} && cp manifest.csv result/ && cd result && zip -q -r ../faceswap_result.zip .
print(open(f"{DRIVE_DIR}/result/faceswap_report.md").read())
print("\nDone: download", f"{DRIVE_DIR}/faceswap_result.zip", "into ~/Projects/trubeat/runs/faceswap/")